# Klasifikasi Multikelas: Topik Berita Reuters

**Buku:** *Deep Learning dengan Python* — Bab 3, Memulai dengan Jaringan Saraf

Mengklasifikasikan berita ke dalam 46 topik yang saling eksklusif (*single-label, multiclass*).

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `keras.utils.np_utils.to_categorical` → `keras.utils.to_categorical`.
- Baseline acak dihitung dengan `np.random.default_rng` (dapat direproduksi).
- Perbaikan bug grafik asli: label sumbu-y grafik akurasi tertulis *Loss*.
- Pelatihan ulang memakai seluruh data latih (asli: tetap memakai data parsial).

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

In [ ]:
import numpy as np
from keras.datasets import reuters

(data_latih, label_latih), (data_uji, label_uji) = reuters.load_data(num_words=10000)
print(len(data_latih), "latih |", len(data_uji), "uji | jumlah topik:", len(set(label_latih)))

## Vektorisasi data dan label
Dua cara mengodekan label: *one-hot* (pakai `categorical_crossentropy`) atau bilangan bulat (pakai `sparse_categorical_crossentropy`).

In [ ]:
def vektorisasi(sekuens, dimensi=10000):
    hasil = np.zeros((len(sekuens), dimensi), dtype="float32")
    for i, s in enumerate(sekuens):
        hasil[i, s] = 1.0
    return hasil

x_latih, x_uji = vektorisasi(data_latih), vektorisasi(data_uji)
y_latih = keras.utils.to_categorical(label_latih)   # one-hot (46 kolom)
y_uji = keras.utils.to_categorical(label_uji)

In [ ]:
from keras import layers

def bangun_model(unit_tengah=64):
    model = keras.Sequential([
        keras.Input(shape=(10000,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(unit_tengah, activation="relu"),
        layers.Dense(46, activation="softmax"),
    ])
    model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

x_val, x_sebagian = x_latih[:1000], x_latih[1000:]
y_val, y_sebagian = y_latih[:1000], y_latih[1000:]

model = bangun_model()
riwayat = model.fit(x_sebagian, y_sebagian, epochs=20, batch_size=512,
                    validation_data=(x_val, y_val))

In [ ]:
import matplotlib.pyplot as plt
h = riwayat.history
epoch = range(1, len(h["loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(epoch, h["loss"], "o", label="Latih"); ax[0].plot(epoch, h["val_loss"], "-", label="Validasi")
ax[0].set(xlabel="Epoch", ylabel="Loss"); ax[0].legend()
ax[1].plot(epoch, h["accuracy"], "o", label="Latih"); ax[1].plot(epoch, h["val_accuracy"], "-", label="Validasi")
ax[1].set(xlabel="Epoch", ylabel="Akurasi"); ax[1].legend()
plt.tight_layout(); plt.show()

Overfitting mulai sekitar epoch ke-9. Latih ulang 9 epoch pada seluruh data latih, lalu bandingkan dengan **baseline acak**.

In [ ]:
model = bangun_model()
model.fit(x_latih, y_latih, epochs=9, batch_size=512, verbose=0)
loss, akurasi = model.evaluate(x_uji, y_uji, verbose=0)

rng = np.random.default_rng(0)
acak = rng.permutation(label_uji)
baseline = np.mean(np.array(label_uji) == acak)
print(f"Akurasi uji: {akurasi:.2%} | baseline acak: {baseline:.2%}")

## Label bilangan bulat
Loss `sparse_categorical_crossentropy` identik secara matematis, hanya antarmukanya berbeda.

In [ ]:
model_sparse = bangun_model()
model_sparse.compile(optimizer="rmsprop",
                     loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_sparse.fit(x_latih, np.array(label_latih), epochs=9, batch_size=512, verbose=0)
print("Akurasi uji (label bulat):", model_sparse.evaluate(x_uji, np.array(label_uji), verbose=0)[1])

## Bottleneck informasi
Lapisan tengah 4 unit terlalu sempit untuk memisahkan 46 kelas.

In [ ]:
model_sempit = bangun_model(unit_tengah=4)
r = model_sempit.fit(x_sebagian, y_sebagian, epochs=20, batch_size=128,
                     validation_data=(x_val, y_val), verbose=0)
print(f"Akurasi validasi terbaik (4 unit): {max(r.history['val_accuracy']):.2%}")